# 

In [ ]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
aqsol_curated_data_path = '/data/data/AqSolDB/results/data_curated.csv'

# Load the curated data
df = pd.read_csv(aqsol_curated_data_path)
df

,ID,Name,InChI,InChIKey,SMILES,Solubility,SD,Occurrences,Group,MolWt,...,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT
0,A-3,"N,N,N-trimethyloctadecan-1-aminium bromide",InChI=1S/C21H46N.BrH/c1-5-6-7-8-9-10-11-12-13-...,SZEMGTQCPRNXEG-UHFFFAOYSA-M,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127,0.000000,1,G1,392.510,...,17.0,142.0,0.0,0.0,0.0,0.0,0.00,158.520601,0.000000e+00,210.377334
1,A-4,Benzo[cd]indol-2(1H)-one,InChI=1S/C11H7NO/c13-11-8-5-1-3-7-4-2-6-9(12-1...,GPYLCFQEKPUWLD-UHFFFAOYSA-N,O=C1Nc2cccc3cccc1c23,-3.254767,0.000000,1,G1,169.183,...,0.0,62.0,2.0,0.0,1.0,3.0,29.10,75.183563,2.582996e+00,511.229248
2,A-5,4-chlorobenzaldehyde,InChI=1S/C7H5ClO/c8-7-3-1-6(5-9)2-4-7/h1-5H,AVPYQKSLYISFPO-UHFFFAOYSA-N,Clc1ccc(C=O)cc1,-2.177078,0.000000,1,G1,140.569,...,1.0,46.0,1.0,0.0,0.0,1.0,17.07,58.261134,3.009782e+00,202.661065
3,A-8,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",InChI=1S/2C23H22O3.Zn/c2*1-15(17-9-5-3-6-10-17...,XTUPUYCJWKHGSW-UHFFFAOYSA-L,[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409,0.000000,1,G1,756.226,...,10.0,264.0,6.0,0.0,0.0,6.0,120.72,323.755434,2.322963e-07,1964.648666
4,A-9,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,InChI=1S/C25H30N2O4/c1-5-20(26(10-22-14-28-22)...,FAUAZXVRLVIARB-UHFFFAOYSA-N,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065,0.000000,1,G1,422.525,...,12.0,164.0,2.0,4.0,4.0,6.0,56.60,183.183268,1.084427e+00,769.899934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9977,I-84,tetracaine,InChI=1S/C15H24N2O2/c1-4-5-10-16-14-8-6-13(7-9...,GKCBAIGFKIBETG-UHFFFAOYSA-N,C(c1ccc(cc1)NCCCC)(=O)OCCN(C)C,-3.010000,0.000000,1,G1,264.369,...,8.0,106.0,1.0,0.0,0.0,1.0,41.57,115.300645,2.394548e+00,374.236893
9978,I-85,tetracycline,InChI=1S/C22H24N2O8/c1-21(31)8-5-4-6-11(25)12(...,OFVLGDICTFRJMM-WESIUVDSSA-N,OC1=C(C(C2=C(O)[C@@](C(C(C(N)=O)=C(O)[C@H]3N(C...,-2.930000,0.000000,1,G1,444.440,...,2.0,170.0,1.0,0.0,3.0,4.0,181.62,182.429237,2.047922e+00,1148.584975
9979,I-86,thymol,InChI=1S/C10H14O/c1-7(2)9-5-4-8(3)6-10(9)11/h4...,MGSRCZKZVOBKFT-UHFFFAOYSA-N,c1(cc(ccc1C(C)C)C)O,-2.190000,0.019222,3,G5,150.221,...,1.0,60.0,1.0,0.0,0.0,1.0,20.23,67.685405,3.092720e+00,251.049732
9980,I-93,verapamil,"InChI=1S/C27H38N2O4/c1-20(2)27(19-28,22-10-12-...",SGTNSNPWRIOYBX-UHFFFAOYSA-N,COc1ccc(CCN(C)CCCC(C#N)(C(C)C)c2ccc(OC)c(OC)c2...,-3.980000,0.000000,1,G1,454.611,...,13.0,180.0,2.0,0.0,0.0,2.0,63.95,198.569223,2.023333e+00,938.203977


In [3]:
df['Group'].value_counts()

Group
G1    7746
G3    1182
G5     636
G2     235
G4     183
Name: count, dtype: int64

In [4]:
# get data instances including G in ID
df_esol = df[df['ID'].str.contains('G')]

# get df_a of instances whose InChI is not in entire df_esol
df_filtered = df[~df['InChI'].isin(df_esol['InChI'])]


In [5]:
dfg1 = df_filtered[df_filtered['Group'] == 'G1']
dfg3 = df_filtered[df_filtered['Group'] == 'G3'] # sd <= 0.5
dfg5 = df_filtered[df_filtered['Group'] == 'G5'] # sd <= 0.5

dfg2 = df_filtered[df_filtered['Group'] == 'G2'] # sd >= 0.5
dfg4 = df_filtered[df_filtered['Group'] == 'G4'] # sd >= 0.5

In [6]:
df.columns

Index(['ID', 'Name', 'InChI', 'InChIKey', 'SMILES', 'Solubility', 'SD',
       'Occurrences', 'Group', 'MolWt', 'MolLogP', 'MolMR', 'HeavyAtomCount',
       'NumHAcceptors', 'NumHDonors', 'NumHeteroatoms', 'NumRotatableBonds',
       'NumValenceElectrons', 'NumAromaticRings', 'NumSaturatedRings',
       'NumAliphaticRings', 'RingCount', 'TPSA', 'LabuteASA', 'BalabanJ',
       'BertzCT'],
      dtype='object')

In [7]:
sd_threshold = 0.1

sd_filtered_df = df[df['SD'] < sd_threshold]


sdfiltered_dfg1 = dfg1[dfg1['SD'] < sd_threshold]
sdfiltered_dfg3 = dfg3[dfg3['SD'] < sd_threshold]
sdfiltered_dfg5 = dfg5[dfg5['SD'] < sd_threshold]

In [8]:
df_esol['Solubility']
# get mean and std of solubility
mean = df_esol['Solubility'].mean()
std = df_esol['Solubility'].std()
mean, std

(np.float64(-3.4133741007194245), np.float64(2.2196316095746425))

In [9]:
mean = sdfiltered_dfg1['Solubility'].mean()
std = sdfiltered_dfg1['Solubility'].std()
mean, std

(np.float64(-2.857196716019586), np.float64(2.3776890074733186))

In [10]:
mean = sdfiltered_dfg3['Solubility'].mean()
std = sdfiltered_dfg3['Solubility'].std()
mean, std

(np.float64(-2.4467583896181404), np.float64(2.13262516060202))

In [11]:
mean = sdfiltered_dfg5['Solubility'].mean()
std = sdfiltered_dfg5['Solubility'].std()
mean, std

(np.float64(-2.202683382360966), np.float64(1.9969950540496824))

In [12]:
# concat sdfiltered_dfg1, sdfiltered_dfg3, sdfiltered_dfg5
concated_filtered_df = pd.concat([sdfiltered_dfg3, sdfiltered_dfg5])

In [13]:
concated_filtered_df

,ID,Name,InChI,InChIKey,SMILES,Solubility,SD,Occurrences,Group,MolWt,...,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT
40,A-58,1-tert-butyl-4-methylbenzene,"InChI=1S/C11H16/c1-9-5-7-10(8-6-9)11(2,3)4/h5-...",QCWXDVFBZVHKLV-UHFFFAOYSA-N,Cc1ccc(cc1)C(C)(C)C,-4.472022,0.048455,2,G3,148.249,...,0.0,60.0,1.0,0.0,0.0,1.0,0.00,69.256114,2.993755,223.210377
50,A-73,S-ethyl dipropylthiocarbamate,InChI=1S/C9H19NOS/c1-4-7-10(8-5-2)9(11)12-6-3/...,GUVLYNGULCJVDO-UHFFFAOYSA-N,CCCN(CCC)C(=O)SCC,-2.703174,0.018736,2,G3,189.324,...,5.0,72.0,0.0,0.0,0.0,0.0,20.31,79.703254,3.810795,121.696943
69,A-114,Prednisolone,InChI=1S/C21H28O5/c1-19-7-5-13(23)9-12(19)3-4-...,OIGNJSKKLXVSLS-VWUMJDOOSA-N,C[C@]12C[C@H](O)[C@H]3[C@@H](CCC4=CC(=O)C=C[C@...,-3.178447,0.015047,2,G3,360.450,...,2.0,142.0,0.0,3.0,4.0,4.0,94.83,153.341308,1.747281,723.913082
142,A-237,2-amino-4-nitrophenol,InChI=1S/C6H6N2O3/c7-5-3-4(8(10)11)1-2-6(5)9/h...,VLZVIIYRNMWPSN-UHFFFAOYSA-N,Nc1cc(ccc1O)[N+]([O-])=O,-2.205602,0.008049,2,G3,154.125,...,1.0,58.0,1.0,0.0,0.0,1.0,89.39,62.218503,3.151550,297.878912
156,A-262,octan-2-one,"InChI=1S/C8H16O/c1-3-4-5-6-7-8(2)9/h3-7H2,1-2H3",ZPVFWPFBNIEHGJ-UHFFFAOYSA-N,CCCCCCC(C)=O,-2.153696,0.051848,2,G3,128.215,...,5.0,54.0,0.0,0.0,0.0,0.0,17.07,57.455368,2.828868,76.636821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9938,I-6,acetazolamide,InChI=1S/C4H6N4O3S2/c1-2(9)6-3-7-8-4(12-3)13(5...,BZKPWHYZMXOIDC-UHFFFAOYSA-N,S(=O)(=O)(N)c1sc(NC(=O)C)nn1,-2.440000,0.059231,3,G5,222.251,...,2.0,72.0,1.0,0.0,0.0,1.0,115.04,78.021451,2.938691,422.352468
9947,I-20,chlorpromazine,InChI=1S/C17H19ClN2S/c1-19(2)10-5-11-20-14-6-3...,ZPEIMTDSQAKGNT-UHFFFAOYSA-N,c1c(Cl)ccc2Sc3ccccc3N(CCCN(C)C)c12,-5.070000,0.036394,3,G5,318.873,...,4.0,110.0,2.0,0.0,1.0,3.0,6.48,135.253420,1.943763,642.437277
9967,I-66,phenobarbital,InChI=1S/C12H12N2O3/c1-2-12(8-6-4-3-5-7-8)9(15...,DDBREPKUVSBGFI-UHFFFAOYSA-N,C1(NC(C(c2ccccc2)(C(=O)N1)CC)=O)=O,-2.290000,0.095670,5,G5,232.239,...,2.0,88.0,1.0,1.0,1.0,2.0,75.27,98.199515,2.535973,461.783925
9968,I-70,pindolol,InChI=1S/C14H20N2O2/c1-10(2)16-8-11(17)9-18-14...,JZQKKSLKJUAGIC-UHFFFAOYSA-N,CC(C)NCC(O)COc1cccc2[nH]ccc12,-3.790000,0.094634,3,G5,248.326,...,6.0,98.0,2.0,0.0,0.0,2.0,57.28,106.969522,1.914310,493.254578


In [14]:
list_label = concated_filtered_df['Solubility'].values
list_smile = concated_filtered_df['SMILES'].values
list_mol = [Chem.MolFromSmiles(smiles) for smiles in list_smile]

[04:45:09] WARNING: not removing hydrogen atom without neighbors
[04:45:09] WARNING: not removing hydrogen atom without neighbors
[04:45:09] WARNING: not removing hydrogen atom without neighbors
[04:45:09] WARNING: not removing hydrogen atom without neighbors
[04:45:09] WARNING: not removing hydrogen atom without neighbors
[04:45:09] WARNING: not removing hydrogen atom without neighbors


In [15]:
list_aqsol_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task="aqsol_logs",
    instruction_templates=instructions_smol.property_prediction_esol_used_later,
)

100%|██████████| 925/925 [00:00<00:00, 1381.21it/s]


In [ ]:
testset_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_augmented'
# load testset
testset = datasets.load_from_disk(testset_path)
esol_testset = testset.filter(lambda x: 'esol' in x['task'])
any('What value' in i for i in esol_testset['prompt_text'])

In [25]:
aqsol_dataset = datasets.Dataset.from_list(list_aqsol_data)
aqsol_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol_0212")

Saving the dataset (1/1 shards): 100%|██████████| 925/925 [00:00<00:00, 16546.11 examples/s]


In [24]:
aqsol_dataset[0]

{'task': 'aqsol_logs',
 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 4, 5, 0, 0, 2, 0, 0],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 4, 5, 3, 0, 2, 0, 0]],
 'edge_index': [[0,
   1,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   4,
   7,
   7,
   8,
   7,
   9,
   7,
   10,
   6,
   1],
  [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 4, 8, 7, 9, 7, 10, 7, 1, 6]],
 'edge_attr': [[0, 0, 0],
  [0, 0, 0],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [3, 0, 1],
  [3, 0, 1]],
 'additional_x': [[5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
 